# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step workflow for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. All dataset structures (record sets, fields, columns) are referenced by their `@id` identifiers for transparency and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
dataset_md = dataset.metadata
print(f"{dataset_md.name}: {dataset_md.description}")
print(f"Dataset fields: {dataset_md.fields}")
print(f"Conforms to: {dataset_md.conforms_to}")

## 2. Data Overview
Review available record sets, fields, and their IDs. We use the Croissant schema to enumerate record sets and inspect their structures. All references to record sets/fields use their `@id` values as required.

**Note:** The FAIR² dataset often uses a single main record set for tabular data. Let's list available record sets and fields with their `@id`s.

In [ ]:
# List all record sets and their fields by @id
record_sets = list(dataset.metadata.record_sets)
print('Available Record Sets:')
for rs in record_sets:
    print(f"- Record Set @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")
    # List fields within this record set
    fields = rs.get('fields', [])
    for field in fields:
        print(f"    - Field @id: {field['@id']}, Name: {field.get('name', '')}, DataType: {field.get('dataType', '')}")

# If records are present, display a sample
if len(record_sets) > 0:
    sample_rs_id = record_sets[0]['@id']
    print(f"\nSample records from record set {sample_rs_id}:")
    for i, record in enumerate(dataset.records(record_set=sample_rs_id)):
        print(record)
        if i >= 2:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references use the explicit `@id` of the record set and fields as discovered in the previous section.

In [ ]:
# Example: extract the main record set (likely the first one)
from pprint import pprint

main_record_set_id = record_sets[0]['@id']
print(f"Extracting data from record set ID: {main_record_set_id}")

records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print(f"Columns in DataFrame (field @ids): \n{df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All references to fields by their `@id` values.

Select a numeric field for analysis— for example, peptide count, molecular weight, or protein coverage. For this demonstration, we'll attempt to use a typical identifier for peptide count (e.g., `'peptide_count'`, but you may need to replace with actual `@id` from the listing above if different).

In [ ]:
# Identify a numeric field (replace below with the actual field @id as enumerated earlier)
sample_numeric_fields = [col for col in df.columns if ('count' in col or 'coverage' in col or 'weight' in col or 'abundance' in col or col.endswith('_number') or col.endswith('_int') or col.endswith('_float'))]
pprint(f"Detected numeric fields (by @id): {sample_numeric_fields}")

# Use the first detected numeric field
if sample_numeric_fields:
    numeric_field = sample_numeric_fields[0]
else:
    raise ValueError('No numeric field found for analysis!')

threshold = df[numeric_field].astype(float).mean()  # use mean as threshold for demonstration
filtered_df = df[df[numeric_field].astype(float) > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[numeric_field + '_normalized'] = (
    filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
) / filtered_df[numeric_field].astype(float).std()

print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

# Group by a key string field, e.g., 'accession' or 'description' by @id
group_candidates = [col for col in df.columns if col.lower().startswith('accession') or 'desc' in col.lower() or col.lower().endswith('id')]
if group_candidates:
    group_field = group_candidates[0]
    grouped = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped by {group_field} (showing means):")
    print(grouped.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we show distribution of the selected numeric field and correlation with another numeric column if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].astype(float), kde=True)
plt.xlabel(numeric_field + ' (@id)')
plt.title(f'Distribution of {numeric_field}')
plt.show()

# If another numeric field is present, show a scatter plot
if len(sample_numeric_fields) > 1:
    plt.figure(figsize=(7,5))
    sns.scatterplot(
        x=df[sample_numeric_fields[0]].astype(float),
        y=df[sample_numeric_fields[1]].astype(float),
        alpha=0.7
    )
    plt.xlabel(sample_numeric_fields[0] + ' (@id)')
    plt.ylabel(sample_numeric_fields[1] + ' (@id)')
    plt.title(f'{sample_numeric_fields[1]} vs. {sample_numeric_fields[0]}')
    plt.show()

## 6. Conclusion
- We loaded metadata and main record set(s) from the FAIR² dataset using `mlcroissant`, referencing all structures by their `@id` fields.
- We listed all available record sets and their fields for transparency.
- Basic EDA and data normalization was performed on the most suitable numeric field.
- Visualizations illustrated data distributions to aid in downstream biological or proteomic analysis.

For custom analyses, substitute `@id` references with those of features/columns relevant to your model or research question.